# Liquidity Predictor - Applied Machine Learning Demo

This notebook builds a machine learning regression algorithm to predict corporate liquidity using financial data from public companies. The model is trained on real data used by investment banking teams to advise clients like Macy's, McDonald's, and Tiffany & Co.

## Section 1: Data Exploration

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

liquidity_data = pd.read_csv('liquidity_data.csv')
liquidity_data.head(3)

In [ ]:
liquidity_data.describe()

## Section 2: Splitting Data

In [ ]:
from sklearn.model_selection import train_test_split

target = liquidity_data.available_liquidity
inputs = liquidity_data.drop('available_liquidity', axis=1)

In [ ]:
target.head(1)

In [ ]:
inputs.head(1)

In [ ]:
results = train_test_split(inputs, target, test_size=0.2, random_state=1)
input_train, input_test, target_train, target_test = results

print(f"Training inputs: {input_train.shape}")
print(f"Testing inputs: {input_test.shape}")
print(f"Training target: {target_train.shape}")
print(f"Testing target: {target_test.shape}")

## Section 3: Model Pipelines

In [ ]:
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pipelines = {
    'lasso': make_pipeline(StandardScaler(), Lasso(random_state=1)),
    'ridge': make_pipeline(StandardScaler(), Ridge(random_state=1)),
    'enet': make_pipeline(StandardScaler(), ElasticNet(random_state=1)),
    'rf': make_pipeline(StandardScaler(), RandomForestRegressor(random_state=1)),
    'gb': make_pipeline(StandardScaler(), GradientBoostingRegressor(random_state=1))
}

for key, value in pipelines.items():
    print(key, type(value))

## Section 4: Hyperparameter Tuning

In [ ]:
lasso_hyperparameters = {
    'lasso__alpha': [0.01, 0.05, 0.1, 0.5, 1, 5]
}

ridge_hyperparameters = {
    'ridge__alpha': [0.01, 0.05, 0.1, 0.5, 1, 5]
}

enet_hyperparameters = {
    'elasticnet__alpha': [0.01, 0.05, 0.1, 0.5, 1, 5],
    'elasticnet__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

rf_hyperparameters = {
    'randomforestregressor__n_estimators': [100, 200],
    'randomforestregressor__max_features': ['auto', 0.3, 0.6]
}

gb_hyperparameters = {
    'gradientboostingregressor__n_estimators': [100, 200],
    'gradientboostingregressor__learning_rate': [0.05, 0.1, 0.2],
    'gradientboostingregressor__max_depth': [1, 3, 5]
}

hyperparameter_grids = {
    'lasso': lasso_hyperparameters,
    'ridge': ridge_hyperparameters,
    'enet': enet_hyperparameters,
    'rf': rf_hyperparameters,
    'gb': gb_hyperparameters
}

## Section 5: Cross-Validation

In [ ]:
from sklearn.model_selection import GridSearchCV

models = {}

for key in pipelines.keys():
    models[key] = GridSearchCV(pipelines[key], hyperparameter_grids[key], cv=5)

for key in models.keys():
    models[key].fit(input_train, target_train)
    print(key, 'is trained and tuned.')

## Section 6: Selecting a Winning Model

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error

for key in models.keys():
    preds = models[key].predict(input_test)
    print(key)
    print('R-Squared: ', round(r2_score(target_test, preds), 3))
    print('MAE: ', round(mean_absolute_error(target_test, preds), 3))
    print('---')

## Section 7: Visualizing Model Predictions

In [ ]:
preds = models['gb'].predict(input_test)

plt.scatter(preds, target_test)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Gradient Boosting: Predicted vs Actual Liquidity')
plt.show()

## Section 8: Using Your Model

In [ ]:
client = pd.read_csv('liquidity_client.csv')
prediction = models['gb'].predict(client)
print(f"Predicted Available Liquidity: ${prediction[0]:,.2f}")